# Drug AI — Phase 1: Research Paper Analysis

A prototype pipeline for PDF text extraction, cleaning, reference filtering, sentence-aware chunking, TF-IDF retrieval, and extractive question answering with page-level evidence.

**Scientific caution:** This is a research-assistance prototype, not a clinical decision-support system. It does not establish drug efficacy, safety, or treatment recommendations.

## 1. Imports and environment

Required packages: `pymupdf`, `nltk`, `scikit-learn`, `torch`, and `transformers`.

Use the project's existing virtual environment.

In [ ]:
import re
import fitz
import nltk
import torch

from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. PDF extraction

Change `PDF_PATH` to the location of the research paper PDF. Page numbers are 1-based.

In [ ]:
PDF_PATH = "../data/research_paper.pdf"  # Change this path

def extract_pdf_pages(pdf_path):
    pages = []
    with fitz.open(pdf_path) as document:
        for page_number, page in enumerate(document, start=1):
            pages.append({"page": page_number, "text": page.get_text("text")})
    return pages

raw_pages = extract_pdf_pages(PDF_PATH)
print("Pages extracted:", len(raw_pages))
print(raw_pages[0]["text"][:1000] if raw_pages else "No pages found")

## 3. Text cleaning

This first-pass cleaner normalizes whitespace and common PDF line-break artifacts. It does not rewrite scientific content.

In [ ]:
def clean_page_text(text):
    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
    text = text.replace("‐", "-").replace("‑", "-")
    text = re.sub(r"-\s*\n\s*", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

cleaned_pages = [
    {"page": item["page"], "text": clean_page_text(item["text"])}
    for item in raw_pages
]
print("Cleaned pages:", len(cleaned_pages))

## 4. Reference filtering

The current paper uses a `References [1]` heading. This filter is tailored to that layout, not a universal reference detector. Inspect the retained text before relying on it.

In [ ]:
def filter_references(pages):
    filtered = []
    references_started = False

    for item in pages:
        text = item["text"]
        if references_started:
            continue

        match = re.search(r"\bReferences\s*\[1\]", text, flags=re.IGNORECASE)
        if match:
            text = text[:match.start()]
            references_started = True

        if text.strip():
            filtered.append({"page": item["page"], "text": text.strip()})

    return filtered

filtered_pages = filter_references(cleaned_pages)
print("Pages retained:", sorted({p["page"] for p in filtered_pages}))

## 5. Sentence-aware chunking

Chunks combine whole sentences up to `MAX_CHARS`. An unusually long sentence is kept intact.

In [ ]:
def create_sentence_chunks(pages, max_chars=900):
    chunks = []
    for page in pages:
        sentences = sent_tokenize(page["text"])
        current = []

        for sentence in sentences:
            candidate = " ".join(current + [sentence]).strip()
            if current and len(candidate) > max_chars:
                chunks.append({"page": page["page"], "text": " ".join(current).strip()})
                current = [sentence]
            else:
                current.append(sentence)

        if current:
            chunks.append({"page": page["page"], "text": " ".join(current).strip()})

    return chunks

chunks = create_sentence_chunks(filtered_pages, max_chars=900)
print("Total chunks:", len(chunks))
print("Pages represented:", sorted({c["page"] for c in chunks}))

## 6. TF-IDF retrieval

The vectorizer is fitted once and reused. Similarity scores indicate lexical similarity, not evidence quality.

In [ ]:
def fit_retriever(chunks):
    texts = [chunk["text"] for chunk in chunks]
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(texts)
    return vectorizer, matrix

vectorizer, chunk_matrix = fit_retriever(chunks)

def retrieve_ranked_evidence(query, chunks, vectorizer, chunk_matrix, top_k=5):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, chunk_matrix).flatten()
    ranked_indices = scores.argsort()[::-1][:top_k]

    results = []
    for index in ranked_indices:
        if scores[index] > 0:
            results.append({
                "page": chunks[index]["page"],
                "text": chunks[index]["text"],
                "score": float(scores[index])
            })
    return results

question = "Which intervention is being evaluated to prevent anti-tuberculosis drug-induced liver injury?"
retrieved = retrieve_ranked_evidence(question, chunks, vectorizer, chunk_matrix, top_k=5)

for item in retrieved:
    print(f"Page {item['page']} | score={item['score']:.4f}")
    print(item["text"][:400], "\n")

## 7. Load the QA model

The first run may download model files. CPU inference is supported, but can be slower.

In [ ]:
MODEL_NAME = "deepset/roberta-base-squad2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Model loaded on:", device)

## 8. Extract an answer span

The score is a heuristic product of start/end softmax values. It is **not a calibrated probability** or a measure of scientific certainty.

In [ ]:
def extract_answer(question, context, tokenizer, model):
    encoded = tokenizer(
        question,
        context,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=512
    )

    offsets = encoded.pop("offset_mapping")[0]
    sequence_ids = encoded.sequence_ids(0)

    device = next(model.parameters()).device
    model_inputs = {key: value.to(device) for key, value in encoded.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**model_inputs)

    start_probs = torch.softmax(outputs.start_logits[0], dim=0)
    end_probs = torch.softmax(outputs.end_logits[0], dim=0)

    context_positions = [i for i, seq_id in enumerate(sequence_ids) if seq_id == 1]

    best_score = -1.0
    best_start = None
    best_end = None

    for start in context_positions:
        for end in context_positions:
            if end < start or end - start > 30:
                continue
            score = start_probs[start].item() * end_probs[end].item()
            if score > best_score:
                best_score, best_start, best_end = score, start, end

    if best_start is None:
        return {"answer": "", "score": 0.0}

    start_char = offsets[best_start][0].item()
    end_char = offsets[best_end][1].item()

    return {
        "answer": context[start_char:end_char].strip(),
        "score": float(best_score)
    }

## 9. Answer with evidence and basic rejection

`0.01` is an experimental threshold only. It is not scientifically validated and may reject valid answers or accept incorrect ones.

In [ ]:
QA_REJECTION_THRESHOLD = 0.01  # Experimental; validate before relying on it

def answer_question_with_qa(
    question, chunks, vectorizer, chunk_matrix, tokenizer, model, top_k=5
):
    evidence = retrieve_ranked_evidence(
        question, chunks, vectorizer, chunk_matrix, top_k=top_k
    )

    if not evidence:
        return {
            "answer": None,
            "source_page": None,
            "evidence": [],
            "message": "No relevant evidence found."
        }

    candidates = []
    for item in evidence:
        qa_result = extract_answer(question, item["text"], tokenizer, model)
        answer = qa_result.get("answer", "").strip()

        if answer:
            candidates.append({
                "answer": answer,
                "qa_score": qa_result["score"],
                "page": item["page"],
                "context": item["text"],
                "retrieval_score": item["score"]
            })

    if not candidates:
        return {
            "answer": None,
            "source_page": None,
            "evidence": evidence,
            "message": "No answer could be extracted."
        }

    best = max(candidates, key=lambda item: item["qa_score"])

    if best["qa_score"] < QA_REJECTION_THRESHOLD:
        return {
            "answer": "Insufficient evidence",
            "source_page": None,
            "qa_score": best["qa_score"],
            "retrieval_score": best["retrieval_score"],
            "evidence": evidence
        }

    return {
        "answer": best["answer"],
        "source_page": best["page"],
        "qa_score": best["qa_score"],
        "retrieval_score": best["retrieval_score"],
        "supporting_text": best["context"],
        "evidence": evidence
    }

## 10. Test: answerable question

Expected answer for this protocol: **milk thistle**. This identifies the intervention being evaluated; it does not establish clinical effectiveness.

In [ ]:
valid_question = (
    "Which intervention is being evaluated to prevent "
    "anti-tuberculosis drug-induced liver injury?"
)

valid_result = answer_question_with_qa(
    valid_question, chunks, vectorizer, chunk_matrix, tokenizer, model, top_k=5
)
print(valid_result)

## 11. Test: unrelated question

This question is outside the paper's subject. The prototype should ideally reject it, but the heuristic threshold does not guarantee reliable no-answer detection.

In [ ]:
unrelated_question = "Who won the Nobel Prize in Physics in 2020?"

unrelated_result = answer_question_with_qa(
    unrelated_question, chunks, vectorizer, chunk_matrix, tokenizer, model, top_k=5
)
print(unrelated_result)

## 12. Scientific interpretation and known limitations

- The source paper is a **protocol for a systematic review and meta-analysis**, not a completed review reporting pooled results.
- It describes a planned evaluation of milk thistle; this paper alone does not establish that milk thistle prevents anti-tuberculosis drug-induced liver injury.
- TF-IDF similarity is lexical relevance, not scientific evidence quality.
- The QA span score is not calibrated confidence.
- The reference filter is tailored to the current PDF layout.
- This prototype does not yet perform robust no-answer detection, contradiction checking, study-quality assessment, or clinical fact verification.

**Next milestone:** create a small manually labelled evaluation set of answerable and unanswerable questions. Measure retrieval and answer quality before tuning thresholds.